# 07 — Dataset Validation, Artifact Analysis, and Robustness

This notebook consolidates the reliability experiments used to support:

> **A Dual Representation Framework for Malicious QR Code Detection Using Fused Feature Learning and Deep Visual Modeling**

It combines three source workflows:

- sanity checks from the main QR experiment notebook;
- dataset artifact analysis from `QR_Dataset_Artifact_Analysis`;
- MobileNetV2 corruption robustness evaluation from the strengthened MobileNetV2 notebook.

## Sections

1. Controlled split loading and path validation
2. Exact duplicate audit using MD5
3. Internal and cross-split duplicate checks
4. Near-duplicate audit using perceptual hashing
5. Label-shuffle diagnostic
6. Metadata-only baseline
7. QR artifact feature extraction
8. Artifact distribution and association tests
9. Artifact-only classifiers
10. MobileNetV2 robustness under realistic degradations
11. Consolidated reliability report

## 1. Imports and Reproducibility

Required optional packages:

```bash
pip install imagehash lightgbm scipy
```

In [ ]:
import hashlib
import json
import os
import random
import time
from pathlib import Path

import cv2
import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from scipy.stats import chi2_contingency, ks_2samp, mannwhitneyu
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Seed:", SEED)

## 2. Repository Paths

In [ ]:
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name.lower() == "notebooks"
    else CURRENT_DIR
)

PROCESSED_DATA_DIR = PROJECT_ROOT / "Data" / "processed"
FUSED_DIR = PROCESSED_DATA_DIR / "fused"

RESULTS_DIR = PROJECT_ROOT / "Results"
RELIABILITY_DIR = RESULTS_DIR / "validation_and_reliability"
SANITY_DIR = RELIABILITY_DIR / "sanity_checks"
ARTIFACT_DIR = RELIABILITY_DIR / "artifact_analysis"
ROBUSTNESS_DIR = RELIABILITY_DIR / "mobilenetv2_robustness"
FIGURES_DIR = RELIABILITY_DIR / "figures"
TABLES_DIR = RELIABILITY_DIR / "tables"

for directory in [
    RELIABILITY_DIR,
    SANITY_DIR,
    ARTIFACT_DIR,
    ROBUSTNESS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = PROCESSED_DATA_DIR / "train.csv"
VAL_CSV = PROCESSED_DATA_DIR / "val.csv"
TEST_CSV = PROCESSED_DATA_DIR / "test.csv"

MOBILE_MODEL_PATH = (
    PROJECT_ROOT
    / "Models"
    / "mobilenetv2"
    / "best_mobilenetv2_fast_strengthened.keras"
)

print("Project root:", PROJECT_ROOT)
print("Reliability outputs:", RELIABILITY_DIR)

## 3. Load Controlled Splits and Validate Paths

In [ ]:
def load_split(path: Path, split_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required split: {path}. Run Notebook 01 first."
        )

    dataframe = pd.read_csv(path).copy()

    if "image_path" not in dataframe.columns and "image" in dataframe.columns:
        dataframe = dataframe.rename(columns={"image": "image_path"})

    required = {"image_path", "label"}
    missing = required - set(dataframe.columns)
    if missing:
        raise ValueError(
            f"{split_name} split is missing columns: {sorted(missing)}"
        )

    dataframe["image_path"] = (
        dataframe["image_path"].astype(str).str.strip()
    )
    dataframe["label"] = (
        dataframe["label"].astype(str).str.strip().str.lower()
    )

    if "label_id" not in dataframe.columns:
        dataframe["label_id"] = dataframe["label"].map(
            {"benign": 0, "malicious": 1}
        )

    dataframe["label_id"] = pd.to_numeric(
        dataframe["label_id"], errors="coerce"
    )

    if dataframe["label_id"].isna().any():
        raise ValueError(f"{split_name} contains invalid labels.")

    dataframe["label_id"] = dataframe["label_id"].astype(int)
    dataframe["split"] = split_name

    dataframe["absolute_path"] = dataframe["image_path"].map(
        lambda value: str((PROJECT_ROOT / value).resolve())
    )

    exists = dataframe["absolute_path"].map(os.path.exists)
    missing_count = int((~exists).sum())

    if missing_count:
        raise FileNotFoundError(
            f"{split_name} contains {missing_count} missing image files."
        )

    return dataframe.reset_index(drop=True)

train_df = load_split(TRAIN_CSV, "train")
val_df = load_split(VAL_CSV, "val")
test_df = load_split(TEST_CSV, "test")

full_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True,
)

split_summary = (
    full_df.groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
split_summary.to_csv(
    TABLES_DIR / "controlled_split_summary.csv",
    index=False,
)

display(split_summary)
print("Total images:", len(full_df))

## 4. Exact Duplicate Audit Using MD5

MD5 is used here only as a byte-level duplicate detector, not for security.

In [ ]:
def md5_hash(file_path: str, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.md5()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()

MD5_CSV = SANITY_DIR / "md5_hashes.csv"

if MD5_CSV.exists():
    md5_df = pd.read_csv(MD5_CSV)
else:
    md5_df = full_df[
        ["image_path", "absolute_path", "label", "label_id", "split"]
    ].copy()
    md5_df["md5"] = [
        md5_hash(path)
        for path in tqdm(
            md5_df["absolute_path"],
            desc="Computing MD5 hashes",
        )
    ]
    md5_df.to_csv(MD5_CSV, index=False)

duplicate_groups = (
    md5_df.groupby("md5")
    .filter(lambda group: len(group) > 1)
    .sort_values("md5")
)

duplicate_groups.to_csv(
    SANITY_DIR / "exact_duplicate_groups.csv",
    index=False,
)

cross_split_exact = duplicate_groups.groupby("md5").filter(
    lambda group: group["split"].nunique() > 1
)
cross_class_exact = duplicate_groups.groupby("md5").filter(
    lambda group: group["label_id"].nunique() > 1
)

print("Exact duplicate rows:", len(duplicate_groups))
print("Cross-split exact duplicate rows:", len(cross_split_exact))
print("Cross-class exact duplicate rows:", len(cross_class_exact))

## 5. Internal and Cross-Split Duplicate Summary

In [ ]:
duplicate_summary_rows = []

for split_name, group in md5_df.groupby("split"):
    duplicate_summary_rows.append(
        {
            "scope": split_name,
            "rows": len(group),
            "unique_md5": group["md5"].nunique(),
            "duplicate_rows": len(group) - group["md5"].nunique(),
        }
    )

duplicate_summary_rows.append(
    {
        "scope": "all_splits",
        "rows": len(md5_df),
        "unique_md5": md5_df["md5"].nunique(),
        "duplicate_rows": len(md5_df) - md5_df["md5"].nunique(),
    }
)

duplicate_summary_df = pd.DataFrame(duplicate_summary_rows)
duplicate_summary_df.to_csv(
    SANITY_DIR / "exact_duplicate_summary.csv",
    index=False,
)

display(duplicate_summary_df)

## 6. Near-Duplicate Audit Using Perceptual Hashing

By default, pHash is computed for the complete dataset. For quick pipeline checks,
set `PHASH_MAX_SAMPLES` to a smaller integer.

The cross-split search first groups identical perceptual hashes. A Hamming-distance
threshold can be added later for a more expensive approximate-neighbor search.

In [ ]:
PHASH_MAX_SAMPLES = None

if PHASH_MAX_SAMPLES is None:
    phash_source_df = full_df.copy()
else:
    phash_source_df = (
        full_df.groupby(["split", "label_id"], group_keys=False)
        .apply(
            lambda group: group.sample(
                n=min(
                    len(group),
                    max(1, int(PHASH_MAX_SAMPLES) // 6),
                ),
                random_state=SEED,
            )
        )
        .reset_index(drop=True)
        .head(int(PHASH_MAX_SAMPLES))
    )

PHASH_CSV = SANITY_DIR / "perceptual_hashes.csv"

if PHASH_CSV.exists() and PHASH_MAX_SAMPLES is None:
    phash_df = pd.read_csv(PHASH_CSV)
else:
    rows = []

    for row in tqdm(
        phash_source_df.itertuples(index=False),
        total=len(phash_source_df),
        desc="Computing perceptual hashes",
    ):
        try:
            with Image.open(row.absolute_path) as image:
                hash_value = str(
                    imagehash.phash(image.convert("L"))
                )
        except Exception:
            hash_value = None

        rows.append(
            {
                "image_path": row.image_path,
                "label": row.label,
                "label_id": row.label_id,
                "split": row.split,
                "phash": hash_value,
            }
        )

    phash_df = pd.DataFrame(rows)
    phash_df.to_csv(PHASH_CSV, index=False)

valid_phash_df = phash_df.dropna(subset=["phash"]).copy()

identical_phash_groups = (
    valid_phash_df.groupby("phash")
    .filter(lambda group: len(group) > 1)
    .sort_values("phash")
)

cross_split_phash = identical_phash_groups.groupby(
    "phash"
).filter(lambda group: group["split"].nunique() > 1)

cross_class_phash = identical_phash_groups.groupby(
    "phash"
).filter(lambda group: group["label_id"].nunique() > 1)

identical_phash_groups.to_csv(
    SANITY_DIR / "identical_phash_groups.csv",
    index=False,
)

print("Identical pHash rows:", len(identical_phash_groups))
print("Cross-split identical pHash rows:", len(cross_split_phash))
print("Cross-class identical pHash rows:", len(cross_class_phash))

## 7. Label-Shuffle Diagnostic

This diagnostic uses the preprocessed fused feature branch when available.
Training labels are randomly shuffled while validation labels remain unchanged.
A result near chance supports the absence of direct label leakage in the pipeline.

This check is diagnostic and does not replace the primary TabNet experiment.

In [ ]:
TRAIN_FUSED = FUSED_DIR / "train_fused_preprocessed.csv"
VAL_FUSED = FUSED_DIR / "val_fused_preprocessed.csv"

label_shuffle_result = {
    "status": "not_run",
    "accuracy": np.nan,
    "roc_auc": np.nan,
}

if TRAIN_FUSED.exists() and VAL_FUSED.exists():
    train_fused = pd.read_csv(TRAIN_FUSED)
    val_fused = pd.read_csv(VAL_FUSED)

    meta_columns = {"image", "image_path", "label", "label_id"}
    feature_columns = [
        column
        for column in train_fused.columns
        if column not in meta_columns
    ]

    X_train_shuffle = train_fused[
        feature_columns
    ].astype(np.float32)
    y_train_original = train_fused["label_id"].astype(int).to_numpy()

    X_val_shuffle = val_fused[
        feature_columns
    ].astype(np.float32)
    y_val_shuffle = val_fused["label_id"].astype(int).to_numpy()

    rng = np.random.default_rng(SEED)
    y_train_shuffled = rng.permutation(y_train_original)

    shuffle_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        random_state=SEED,
        n_jobs=-1,
    )
    shuffle_model.fit(X_train_shuffle, y_train_shuffled)

    shuffle_pred = shuffle_model.predict(X_val_shuffle)
    shuffle_prob = shuffle_model.predict_proba(
        X_val_shuffle
    )[:, 1]

    label_shuffle_result = {
        "status": "completed",
        "accuracy": accuracy_score(
            y_val_shuffle, shuffle_pred
        ),
        "roc_auc": roc_auc_score(
            y_val_shuffle, shuffle_prob
        ),
    }
else:
    print(
        "Fused feature files were not found; "
        "the label-shuffle diagnostic was skipped."
    )

pd.DataFrame([label_shuffle_result]).to_csv(
    SANITY_DIR / "label_shuffle_result.csv",
    index=False,
)

print(label_shuffle_result)

## 8. Metadata-Only Baseline

This baseline uses file-level and basic dimensional metadata only. It is intended to
test whether simple non-content properties are strongly predictive of the label.

In [ ]:
def metadata_features(dataframe: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for row in tqdm(
        dataframe.itertuples(index=False),
        total=len(dataframe),
        desc="Extracting metadata features",
    ):
        try:
            with Image.open(row.absolute_path) as image:
                width, height = image.size
                channels = len(image.getbands())
        except Exception:
            width, height, channels = np.nan, np.nan, np.nan

        rows.append(
            {
                "image_path": row.image_path,
                "split": row.split,
                "label_id": row.label_id,
                "file_size_bytes": os.path.getsize(
                    row.absolute_path
                ),
                "width": width,
                "height": height,
                "channels": channels,
                "extension_length": len(
                    Path(row.image_path).suffix
                ),
                "filename_length": len(
                    Path(row.image_path).name
                ),
            }
        )

    return pd.DataFrame(rows)

metadata_df = metadata_features(full_df)

metadata_columns = [
    "file_size_bytes",
    "width",
    "height",
    "channels",
    "extension_length",
    "filename_length",
]

metadata_train = metadata_df[
    metadata_df["split"] == "train"
]
metadata_val = metadata_df[
    metadata_df["split"] == "val"
]
metadata_test = metadata_df[
    metadata_df["split"] == "test"
]

metadata_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                min_samples_leaf=2,
                random_state=SEED,
                n_jobs=-1,
            ),
        ),
    ]
)

metadata_pipeline.fit(
    metadata_train[metadata_columns],
    metadata_train["label_id"],
)

metadata_results = []

for split_name, split_df in [
    ("val", metadata_val),
    ("test", metadata_test),
]:
    prediction = metadata_pipeline.predict(
        split_df[metadata_columns]
    )
    probability = metadata_pipeline.predict_proba(
        split_df[metadata_columns]
    )[:, 1]

    metadata_results.append(
        {
            "split": split_name,
            "accuracy": accuracy_score(
                split_df["label_id"], prediction
            ),
            "precision": precision_score(
                split_df["label_id"],
                prediction,
                zero_division=0,
            ),
            "recall": recall_score(
                split_df["label_id"],
                prediction,
                zero_division=0,
            ),
            "f1_score": f1_score(
                split_df["label_id"],
                prediction,
                zero_division=0,
            ),
            "roc_auc": roc_auc_score(
                split_df["label_id"], probability
            ),
        }
    )

metadata_results_df = pd.DataFrame(metadata_results)
metadata_results_df.to_csv(
    SANITY_DIR / "metadata_only_baseline_results.csv",
    index=False,
)

display(metadata_results_df)

# Part II — Dataset Artifact Analysis

The following features are diagnostic indicators of dataset construction and QR
encoding properties. They should not be interpreted as semantic maliciousness features.

## 9. Artifact Feature Extraction Functions

In [ ]:
def decode_qr(gray_image):
    detector = cv2.QRCodeDetector()

    try:
        text, points, _ = detector.detectAndDecode(gray_image)
        return text or "", points
    except Exception:
        return "", None

def estimate_module_count_and_version(gray_image):
    try:
        _, binary = cv2.threshold(
            gray_image,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU,
        )

        dark = binary < 128
        ys, xs = np.where(dark)

        if len(xs) < 10 or len(ys) < 10:
            return np.nan, np.nan

        crop = binary[
            ys.min() : ys.max() + 1,
            xs.min() : xs.max() + 1,
        ]

        height, width = crop.shape

        if height < 20 or width < 20:
            return np.nan, np.nan

        scan_lines = []

        for fraction in [0.35, 0.50, 0.65]:
            scan_lines.append(
                crop[int(height * fraction), :]
            )
            scan_lines.append(
                crop[:, int(width * fraction)]
            )

        run_lengths = []

        for line in scan_lines:
            binary_line = (line < 128).astype(np.uint8)
            changes = np.where(
                np.diff(binary_line) != 0
            )[0]

            if len(changes) < 5:
                continue

            positions = np.r_[
                0,
                changes + 1,
                len(binary_line),
            ]
            runs = np.diff(positions)

            if len(runs) > 0:
                upper = np.percentile(runs, 90)
                runs = runs[
                    (runs > 0) & (runs < upper)
                ]
                run_lengths.extend(runs.tolist())

        if len(run_lengths) < 10:
            return np.nan, np.nan

        module_pixels = np.median(run_lengths)

        if module_pixels <= 0:
            return np.nan, np.nan

        raw_modules = np.mean(
            [height, width]
        ) / module_pixels

        valid_counts = np.array(
            [21 + 4 * (version - 1) for version in range(1, 41)]
        )
        module_count = int(
            valid_counts[
                np.argmin(
                    np.abs(valid_counts - raw_modules)
                )
            ]
        )
        version = int((module_count - 21) / 4 + 1)

        return module_count, version

    except Exception:
        return np.nan, np.nan

def extract_artifact_features(row) -> dict:
    image_path = row.absolute_path

    output = {
        "image_path": row.image_path,
        "split": row.split,
        "label": row.label,
        "label_id": row.label_id,
        "file_size_bytes": (
            os.path.getsize(image_path)
            if os.path.exists(image_path)
            else np.nan
        ),
    }

    try:
        with Image.open(image_path) as image:
            output["width"], output["height"] = image.size
            output["mode"] = image.mode

        gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        if gray is None:
            raise ValueError("Unreadable image")

        height, width = gray.shape
        output["pixel_area"] = width * height
        output["black_pixel_ratio"] = float(
            (gray < 128).mean()
        )
        output["mean_intensity"] = float(gray.mean())
        output["std_intensity"] = float(gray.std())

        edges = cv2.Canny(gray, 100, 200)
        output["edge_density"] = float(
            (edges > 0).mean()
        )

        decoded_text, points = decode_qr(gray)
        output["decode_success"] = int(
            bool(decoded_text)
        )
        output["payload_length"] = len(decoded_text)
        output["num_digits"] = sum(
            character.isdigit()
            for character in decoded_text
        )
        output["num_letters"] = sum(
            character.isalpha()
            for character in decoded_text
        )
        output["num_special_chars"] = sum(
            not character.isalnum()
            for character in decoded_text
        )
        output["num_slashes"] = decoded_text.count("/")
        output["num_dots"] = decoded_text.count(".")
        output["num_hyphens"] = decoded_text.count("-")
        output["num_query_chars"] = decoded_text.count("?")
        output["is_url_like"] = int(
            decoded_text.lower().startswith(
                ("http://", "https://", "www.")
            )
        )

        if points is not None:
            points_array = np.asarray(points).reshape(-1, 2)
            polygon_area = abs(
                cv2.contourArea(
                    points_array.astype(np.float32)
                )
            )
            output["detected_qr_bbox_ratio"] = float(
                polygon_area / (width * height)
            )
        else:
            output["detected_qr_bbox_ratio"] = np.nan

        module_count, version = (
            estimate_module_count_and_version(gray)
        )
        output["estimated_module_count"] = module_count
        output["estimated_qr_version"] = version

    except Exception:
        for column in [
            "width",
            "height",
            "mode",
            "pixel_area",
            "black_pixel_ratio",
            "mean_intensity",
            "std_intensity",
            "edge_density",
            "decode_success",
            "payload_length",
            "num_digits",
            "num_letters",
            "num_special_chars",
            "num_slashes",
            "num_dots",
            "num_hyphens",
            "num_query_chars",
            "is_url_like",
            "detected_qr_bbox_ratio",
            "estimated_module_count",
            "estimated_qr_version",
        ]:
            output.setdefault(column, np.nan)

    return output

## 10. Run Artifact Extraction

In [ ]:
ARTIFACT_CSV = ARTIFACT_DIR / "qr_dataset_artifact_features.csv"

if ARTIFACT_CSV.exists():
    artifact_df = pd.read_csv(ARTIFACT_CSV)
else:
    artifact_df = pd.DataFrame(
        [
            extract_artifact_features(row)
            for row in tqdm(
                full_df.itertuples(index=False),
                total=len(full_df),
                desc="Extracting artifact features",
            )
        ]
    )
    artifact_df.to_csv(ARTIFACT_CSV, index=False)

print("Artifact feature table:", artifact_df.shape)
display(artifact_df.head())

## 11. Numeric Artifact Summary and Statistical Tests

In [ ]:
numeric_artifact_features = [
    "width",
    "height",
    "pixel_area",
    "file_size_bytes",
    "payload_length",
    "num_digits",
    "num_letters",
    "num_special_chars",
    "num_slashes",
    "num_dots",
    "num_hyphens",
    "num_query_chars",
    "black_pixel_ratio",
    "mean_intensity",
    "std_intensity",
    "edge_density",
    "estimated_module_count",
    "estimated_qr_version",
    "detected_qr_bbox_ratio",
]

available_numeric = [
    column
    for column in numeric_artifact_features
    if column in artifact_df.columns
]

summary = artifact_df.groupby("label")[
    available_numeric
].agg(["count", "mean", "std", "median", "min", "max"])

summary.to_csv(
    ARTIFACT_DIR / "artifact_summary_by_class.csv"
)

benign = artifact_df[
    artifact_df["label_id"] == 0
]
malicious = artifact_df[
    artifact_df["label_id"] == 1
]

statistical_rows = []

for feature in available_numeric:
    benign_values = pd.to_numeric(
        benign[feature], errors="coerce"
    ).dropna()
    malicious_values = pd.to_numeric(
        malicious[feature], errors="coerce"
    ).dropna()

    if len(benign_values) < 2 or len(malicious_values) < 2:
        continue

    mann_stat, mann_p = mannwhitneyu(
        benign_values,
        malicious_values,
        alternative="two-sided",
    )
    ks_stat, ks_p = ks_2samp(
        benign_values,
        malicious_values,
    )

    pooled_std = np.sqrt(
        (
            benign_values.var(ddof=1)
            + malicious_values.var(ddof=1)
        )
        / 2
    )

    standardized_difference = (
        (
            malicious_values.mean()
            - benign_values.mean()
        )
        / pooled_std
        if pooled_std > 0
        else np.nan
    )

    statistical_rows.append(
        {
            "feature": feature,
            "benign_mean": benign_values.mean(),
            "malicious_mean": malicious_values.mean(),
            "benign_median": benign_values.median(),
            "malicious_median": malicious_values.median(),
            "standardized_mean_difference": standardized_difference,
            "mannwhitney_p_value": mann_p,
            "ks_statistic": ks_stat,
            "ks_p_value": ks_p,
        }
    )

artifact_stats_df = pd.DataFrame(
    statistical_rows
).sort_values(
    "ks_statistic",
    ascending=False,
)

artifact_stats_df.to_csv(
    ARTIFACT_DIR / "artifact_statistical_tests.csv",
    index=False,
)

display(artifact_stats_df)

## 12. Categorical Artifact Association Tests

In [ ]:
categorical_features = [
    "mode",
    "width",
    "height",
    "estimated_qr_version",
    "estimated_module_count",
    "decode_success",
    "is_url_like",
]

categorical_rows = []

for feature in categorical_features:
    if feature not in artifact_df.columns:
        continue

    contingency = pd.crosstab(
        artifact_df[feature],
        artifact_df["label"],
    )

    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        continue

    try:
        chi2, p_value, degrees, _ = chi2_contingency(
            contingency
        )

        categorical_rows.append(
            {
                "feature": feature,
                "chi2": chi2,
                "p_value": p_value,
                "degrees_of_freedom": degrees,
                "number_of_categories": contingency.shape[0],
            }
        )
    except Exception:
        pass

categorical_stats_df = pd.DataFrame(
    categorical_rows
)

if not categorical_stats_df.empty:
    categorical_stats_df = categorical_stats_df.sort_values(
        "p_value"
    )

categorical_stats_df.to_csv(
    ARTIFACT_DIR / "artifact_categorical_tests.csv",
    index=False,
)

display(categorical_stats_df)

## 13. Artifact-Only Diagnostic Classifiers

In [ ]:
artifact_model_features = [
    "width",
    "height",
    "pixel_area",
    "file_size_bytes",
    "payload_length",
    "num_digits",
    "num_letters",
    "num_special_chars",
    "num_slashes",
    "num_dots",
    "num_hyphens",
    "num_query_chars",
    "black_pixel_ratio",
    "mean_intensity",
    "std_intensity",
    "edge_density",
    "estimated_module_count",
    "estimated_qr_version",
    "detected_qr_bbox_ratio",
    "decode_success",
    "is_url_like",
]

artifact_model_features = [
    column
    for column in artifact_model_features
    if column in artifact_df.columns
]

model_df = artifact_df[
    ["split", "label_id"] + artifact_model_features
].copy()

for column in artifact_model_features:
    model_df[column] = pd.to_numeric(
        model_df[column],
        errors="coerce",
    )

train_artifact = model_df[
    model_df["split"] == "train"
]
val_artifact = model_df[
    model_df["split"] == "val"
]
test_artifact = model_df[
    model_df["split"] == "test"
]

classifiers = {
    "Artifact-only Logistic Regression": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "Artifact-only Random Forest": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=200,
                    min_samples_leaf=2,
                    random_state=SEED,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

artifact_classifier_results = []

for model_name, classifier in classifiers.items():
    classifier.fit(
        train_artifact[artifact_model_features],
        train_artifact["label_id"],
    )

    for split_name, split_df in [
        ("val", val_artifact),
        ("test", test_artifact),
    ]:
        prediction = classifier.predict(
            split_df[artifact_model_features]
        )
        probability = classifier.predict_proba(
            split_df[artifact_model_features]
        )[:, 1]

        artifact_classifier_results.append(
            {
                "model": model_name,
                "split": split_name,
                "accuracy": accuracy_score(
                    split_df["label_id"], prediction
                ),
                "precision": precision_score(
                    split_df["label_id"],
                    prediction,
                    zero_division=0,
                ),
                "recall": recall_score(
                    split_df["label_id"],
                    prediction,
                    zero_division=0,
                ),
                "f1_score": f1_score(
                    split_df["label_id"],
                    prediction,
                    zero_division=0,
                ),
                "roc_auc": roc_auc_score(
                    split_df["label_id"], probability
                ),
            }
        )

artifact_classifier_results_df = pd.DataFrame(
    artifact_classifier_results
)

artifact_classifier_results_df.to_csv(
    ARTIFACT_DIR / "artifact_only_classifier_results.csv",
    index=False,
)

display(artifact_classifier_results_df)

## 14. Artifact Feature Importance

In [ ]:
random_forest_pipeline = classifiers[
    "Artifact-only Random Forest"
]

random_forest_model = random_forest_pipeline.named_steps[
    "classifier"
]

importance_df = pd.DataFrame(
    {
        "feature": artifact_model_features,
        "importance": random_forest_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

importance_df.to_csv(
    ARTIFACT_DIR / "artifact_feature_importance.csv",
    index=False,
)

top_features = importance_df.head(15).iloc[::-1]

plt.figure(figsize=(8, 6))
plt.barh(
    top_features["feature"],
    top_features["importance"],
)
plt.xlabel("Importance")
plt.title("Top Artifact Features Predicting Class Label")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "artifact_feature_importance.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

# Part III — MobileNetV2 Robustness Evaluation

This section loads the saved MobileNetV2 model from Notebook 04 and evaluates it under
deterministic, realistic image degradations without retraining.

## 15. Load the Saved MobileNetV2 Model

In [ ]:
if not MOBILE_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"MobileNetV2 model not found: {MOBILE_MODEL_PATH}. "
        "Run Notebook 04 first."
    )

mobilenet_model = tf.keras.models.load_model(
    MOBILE_MODEL_PATH
)

IMG_SIZE = 128
ROBUSTNESS_BATCH_SIZE = 256
ROBUSTNESS_THRESHOLD = 0.50
ROBUSTNESS_MAX_SAMPLES = None

if ROBUSTNESS_MAX_SAMPLES is None:
    robustness_df = test_df.copy().reset_index(drop=True)
else:
    robustness_df = (
        test_df.groupby("label_id", group_keys=False)
        .apply(
            lambda group: group.sample(
                n=min(
                    len(group),
                    max(
                        1,
                        int(ROBUSTNESS_MAX_SAMPLES) // 2,
                    ),
                ),
                random_state=SEED,
            )
        )
        .reset_index(drop=True)
        .head(int(ROBUSTNESS_MAX_SAMPLES))
    )

print("Robustness samples:", len(robustness_df))

## 16. Deterministic Corruption Functions

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def gaussian_blur(image, kernel_size=3, sigma=0.3):
    radius = kernel_size // 2
    coordinates = tf.range(
        -radius,
        radius + 1,
        dtype=tf.float32,
    )
    kernel_1d = tf.exp(
        -(coordinates ** 2) / (2.0 * sigma ** 2)
    )
    kernel_1d = kernel_1d / tf.reduce_sum(kernel_1d)
    kernel_2d = tf.tensordot(
        kernel_1d,
        kernel_1d,
        axes=0,
    )
    kernel_2d = kernel_2d[
        :, :, tf.newaxis, tf.newaxis
    ]
    kernel_2d = tf.tile(
        kernel_2d,
        [1, 1, 3, 1],
    )

    return tf.nn.depthwise_conv2d(
        image[tf.newaxis, ...],
        kernel_2d,
        strides=[1, 1, 1, 1],
        padding="SAME",
    )[0]

def jpeg_compress(image, quality):
    image_uint8 = tf.cast(
        tf.clip_by_value(image, 0.0, 255.0),
        tf.uint8,
    )
    encoded = tf.io.encode_jpeg(
        image_uint8,
        quality=int(quality),
        chroma_downsampling=True,
    )
    decoded = tf.io.decode_jpeg(
        encoded,
        channels=3,
    )
    return tf.cast(decoded, tf.float32)

def rotate_image(image, degrees):
    radians = tf.cast(
        degrees * np.pi / 180.0,
        tf.float32,
    )

    cosine = tf.math.cos(radians)
    sine = tf.math.sin(radians)

    height = tf.cast(tf.shape(image)[0], tf.float32)
    width = tf.cast(tf.shape(image)[1], tf.float32)
    center_x = (width - 1.0) / 2.0
    center_y = (height - 1.0) / 2.0

    transform = [
        cosine,
        sine,
        center_x - cosine * center_x - sine * center_y,
        -sine,
        cosine,
        center_y + sine * center_x - cosine * center_y,
        0.0,
        0.0,
    ]

    rotated = tf.raw_ops.ImageProjectiveTransformV3(
        images=image[tf.newaxis, ...],
        transforms=tf.reshape(
            tf.stack(transform),
            [1, 8],
        ),
        output_shape=tf.shape(image)[:2],
        interpolation="BILINEAR",
        fill_mode="CONSTANT",
        fill_value=255.0,
    )

    return rotated[0]

def apply_corruption(image, index, condition):
    if condition == "original":
        return image

    if condition == "gaussian_blur_mild":
        return gaussian_blur(image, 3, 0.3)

    if condition == "gaussian_noise_mild":
        noise = tf.random.stateless_normal(
            tf.shape(image),
            seed=tf.stack(
                [tf.cast(SEED, tf.int32), index]
            ),
            mean=0.0,
            stddev=3.0,
        )
        return tf.clip_by_value(
            image + noise,
            0.0,
            255.0,
        )

    if condition == "jpeg_q90":
        return jpeg_compress(image, 90)

    if condition == "jpeg_q80":
        return jpeg_compress(image, 80)

    if condition == "brightness_minus10":
        return tf.clip_by_value(
            image * 0.90,
            0.0,
            255.0,
        )

    if condition == "brightness_plus10":
        return tf.clip_by_value(
            image * 1.10,
            0.0,
            255.0,
        )

    if condition == "rotation_minus2":
        return rotate_image(image, -2.0)

    if condition == "rotation_plus2":
        return rotate_image(image, 2.0)

    if condition == "resolution_80":
        reduced = tf.image.resize(
            image,
            [
                int(IMG_SIZE * 0.8),
                int(IMG_SIZE * 0.8),
            ],
            method="bilinear",
            antialias=True,
        )
        return tf.image.resize(
            reduced,
            [IMG_SIZE, IMG_SIZE],
            method="bilinear",
            antialias=True,
        )

    raise ValueError(
        f"Unknown robustness condition: {condition}"
    )

## 17. Build Corrupted Test Datasets and Evaluate

In [ ]:
ROBUSTNESS_CONDITIONS = [
    "original",
    "gaussian_blur_mild",
    "gaussian_noise_mild",
    "jpeg_q90",
    "jpeg_q80",
    "brightness_minus10",
    "brightness_plus10",
    "rotation_minus2",
    "rotation_plus2",
    "resolution_80",
]

def make_robustness_dataset(dataframe, condition):
    paths = dataframe["absolute_path"].astype(str).to_numpy()
    labels = dataframe["label_id"].astype(
        "float32"
    ).to_numpy()
    indices = np.arange(
        len(dataframe),
        dtype=np.int32,
    )

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels, indices)
    )

    options = tf.data.Options()
    options.experimental_deterministic = True
    dataset = dataset.with_options(options)

    def load_corrupt_preprocess(path, label, index):
        raw = tf.io.read_file(path)
        image = tf.io.decode_image(
            raw,
            channels=3,
            expand_animations=False,
        )
        image.set_shape([None, None, 3])

        image = tf.image.resize(
            image,
            [IMG_SIZE, IMG_SIZE],
            method="bilinear",
            antialias=True,
        )
        image = tf.cast(image, tf.float32)
        image = apply_corruption(
            image,
            index,
            condition,
        )
        image = tf.keras.applications.mobilenet_v2.preprocess_input(
            image
        )

        return image, tf.cast(label, tf.float32)

    dataset = dataset.map(
        load_corrupt_preprocess,
        num_parallel_calls=AUTOTUNE,
    )
    dataset = dataset.batch(
        ROBUSTNESS_BATCH_SIZE,
        drop_remainder=False,
    )
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

robustness_rows = []
y_true = robustness_df["label_id"].to_numpy(dtype=int)

for condition in ROBUSTNESS_CONDITIONS:
    dataset = make_robustness_dataset(
        robustness_df,
        condition,
    )

    start_time = time.perf_counter()
    probabilities = mobilenet_model.predict(
        dataset,
        verbose=1,
    ).reshape(-1)
    elapsed = time.perf_counter() - start_time

    predictions = (
        probabilities >= ROBUSTNESS_THRESHOLD
    ).astype(int)

    matrix = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    )

    robustness_rows.append(
        {
            "condition": condition,
            "n_samples": len(robustness_df),
            "threshold": ROBUSTNESS_THRESHOLD,
            "accuracy": accuracy_score(
                y_true, predictions
            ),
            "precision": precision_score(
                y_true,
                predictions,
                zero_division=0,
            ),
            "recall": recall_score(
                y_true,
                predictions,
                zero_division=0,
            ),
            "f1_score": f1_score(
                y_true,
                predictions,
                zero_division=0,
            ),
            "roc_auc": roc_auc_score(
                y_true, probabilities
            ),
            "average_precision": average_precision_score(
                y_true, probabilities
            ),
            "inference_seconds": elapsed,
            "true_negatives": int(matrix[0, 0]),
            "false_positives": int(matrix[0, 1]),
            "false_negatives": int(matrix[1, 0]),
            "true_positives": int(matrix[1, 1]),
        }
    )

robustness_results_df = pd.DataFrame(
    robustness_rows
)

original_accuracy = float(
    robustness_results_df.loc[
        robustness_results_df["condition"] == "original",
        "accuracy",
    ].iloc[0]
)

robustness_results_df["accuracy_drop_pp"] = (
    original_accuracy
    - robustness_results_df["accuracy"]
) * 100.0

robustness_results_df.to_csv(
    ROBUSTNESS_DIR / "mobilenetv2_robustness_results.csv",
    index=False,
)

display(robustness_results_df)

## 18. Robustness Publication Table and Figure

In [ ]:
condition_labels = {
    "original": "Original",
    "gaussian_blur_mild": "Mild Gaussian blur",
    "gaussian_noise_mild": "Mild Gaussian noise",
    "jpeg_q90": "JPEG quality 90",
    "jpeg_q80": "JPEG quality 80",
    "brightness_minus10": "Brightness -10%",
    "brightness_plus10": "Brightness +10%",
    "rotation_minus2": "Rotation -2 degrees",
    "rotation_plus2": "Rotation +2 degrees",
    "resolution_80": "80% resolution",
}

robustness_publication = robustness_results_df.copy()
robustness_publication["Condition"] = (
    robustness_publication["condition"].map(
        condition_labels
    )
)

publication_table = robustness_publication[
    [
        "Condition",
        "accuracy",
        "precision",
        "recall",
        "f1_score",
        "roc_auc",
        "accuracy_drop_pp",
    ]
].rename(
    columns={
        "accuracy": "Accuracy",
        "precision": "Precision",
        "recall": "Recall",
        "f1_score": "F1-score",
        "roc_auc": "ROC-AUC",
        "accuracy_drop_pp": "Accuracy drop (pp)",
    }
)

publication_table.to_csv(
    ROBUSTNESS_DIR / "mobilenetv2_robustness_publication_table.csv",
    index=False,
)

plot_df = robustness_publication.set_index(
    "Condition"
)[["accuracy", "f1_score"]]

axis = plot_df.plot(
    kind="bar",
    figsize=(12, 6),
)
axis.set_xlabel("Test condition")
axis.set_ylabel("Score")
axis.set_title(
    "MobileNetV2 Robustness under QR-Image Degradations"
)
axis.set_ylim(0.0, 1.02)
axis.tick_params(axis="x", rotation=35)
axis.legend(["Accuracy", "F1-score"])
axis.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "mobilenetv2_robustness_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

display(publication_table)

## 19. Consolidated Reliability Summary

In [ ]:
reliability_summary = {
    "total_images": int(len(full_df)),
    "exact_duplicate_rows": int(len(duplicate_groups)),
    "cross_split_exact_duplicate_rows": int(
        len(cross_split_exact)
    ),
    "cross_class_exact_duplicate_rows": int(
        len(cross_class_exact)
    ),
    "identical_phash_rows": int(
        len(identical_phash_groups)
    ),
    "cross_split_identical_phash_rows": int(
        len(cross_split_phash)
    ),
    "cross_class_identical_phash_rows": int(
        len(cross_class_phash)
    ),
    "label_shuffle_status": label_shuffle_result[
        "status"
    ],
    "label_shuffle_accuracy": label_shuffle_result[
        "accuracy"
    ],
    "label_shuffle_roc_auc": label_shuffle_result[
        "roc_auc"
    ],
    "metadata_only_test_accuracy": float(
        metadata_results_df.loc[
            metadata_results_df["split"] == "test",
            "accuracy",
        ].iloc[0]
    ),
    "metadata_only_test_roc_auc": float(
        metadata_results_df.loc[
            metadata_results_df["split"] == "test",
            "roc_auc",
        ].iloc[0]
    ),
    "best_artifact_only_test_accuracy": float(
        artifact_classifier_results_df.loc[
            artifact_classifier_results_df["split"] == "test",
            "accuracy",
        ].max()
    ),
    "best_artifact_only_test_roc_auc": float(
        artifact_classifier_results_df.loc[
            artifact_classifier_results_df["split"] == "test",
            "roc_auc",
        ].max()
    ),
    "mobilenet_original_accuracy": original_accuracy,
    "mobilenet_lowest_robustness_accuracy": float(
        robustness_results_df.loc[
            robustness_results_df["condition"] != "original",
            "accuracy",
        ].min()
    ),
}

with (
    RELIABILITY_DIR / "consolidated_reliability_summary.json"
).open("w", encoding="utf-8") as file:
    json.dump(
        reliability_summary,
        file,
        indent=2,
        allow_nan=True,
    )

reliability_summary_df = pd.DataFrame(
    [
        {
            "check": key,
            "value": value,
        }
        for key, value in reliability_summary.items()
    ]
)

reliability_summary_df.to_csv(
    TABLES_DIR / "consolidated_reliability_summary.csv",
    index=False,
)

display(reliability_summary_df)

print("=" * 76)
print("DATASET VALIDATION AND RELIABILITY ANALYSIS COMPLETED")
print("=" * 76)
print("Outputs:", RELIABILITY_DIR)
print("=" * 76)